Firstly the LMC proportions per sample were obtained from the MeDeCom object with this script: library(RnBeads)

rnb <- load.rnb.set("/path/to/data/rnb.set.tum.no.regions.ICA.AED.noNA.1015")

# Get sample information
sample_info <- pheno(rnb)
cat("Sample info dimensions:", dim(sample_info), "\n")
cat("Sample info columns:", colnames(sample_info), "\n")
cat("First 5 rows:\n")
print(head(sample_info, 5))

# Get sample IDs from RnBeads object
sample_ids <- sample_info$Sample_ID
cat("Number of sample IDs:", length(sample_ids), "\n")
cat("First 5:", sample_ids[1:5], "\n")

# Attach as column names to proportions matrix
colnames(props) <- sample_ids

# Verify
cat("Props dimensions:", dim(props), "\n")
cat("First 5 colnames:", colnames(props)[1:5], "\n")

# Save with sample IDs
write.csv(props, "/path/to/data/LMC_proportions_per_sample.csv", quote=FALSE)
cat("Saved CSV with sample IDs\n")

In [ ]:
import pandas as pd
import numpy as np

# Reload metadata
metadata = pd.read_csv('/path/to/data/metadata_common_PPCG.tsv', sep='\t')
metadata['ppcg_id'] = metadata['ppcg_id'].str.strip()

# Load LMC proportions
lmc_props = pd.read_csv('/path/to/data/LMC_proportions_per_sample.csv', index_col=0)

print("LMC props shape:", lmc_props.shape)
print("First 5 columns:", lmc_props.columns[:5].tolist())

# Transpose so samples are rows, LMCs are columns
lmc_props = lmc_props.T
print("After transpose:", lmc_props.shape)

# Extract PPCG ID from sample name — PPCG0001c_450k -> PPCG0001
lmc_props.index = lmc_props.index.str.extract(r'(PPCG\d+)')[0].values
print("After ID extraction, first 5 index:", lmc_props.index[:5].tolist())

# Check overlap with metadata
overlap = len(set(lmc_props.index) & set(metadata['ppcg_id']))
print(f"\nOverlap with metadata: {overlap} patients")

In [ ]:
from scipy.stats import rankdata
from scipy.stats import norm
from lifelines import CoxPHFitter
from lifelines.statistics import logrank_test
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Aggregate to patient level ────────────────────────────────────────
lmc_patient = lmc_props.groupby(lmc_props.index).mean()
print(f"Patient-level LMC matrix: {lmc_patient.shape}")

lmc_cols = lmc_patient.columns.tolist()

# ── Clean metadata ────────────────────────────────────────────────────
clinical_cols = ['relapse_ind', 'donor_relapse_interval',
                 'donor_interval_of_last_followup']

metadata_clean = metadata.copy()
for col in clinical_cols:
    metadata_clean[col] = metadata_clean[col].replace(999, np.nan)

# ── Merge with metadata ───────────────────────────────────────────────
merged_lmc = lmc_patient.merge(
    metadata_clean.set_index('ppcg_id')[clinical_cols],
    left_index=True, right_index=True,
    how='inner'
)
print(f"After merge: {merged_lmc.shape}")

# ── Build survival variables ──────────────────────────────────────────
merged_lmc['time'] = np.where(
    merged_lmc['relapse_ind'] == 1,
    merged_lmc['donor_relapse_interval'],
    merged_lmc['donor_interval_of_last_followup']
)
merged_lmc['event'] = merged_lmc['relapse_ind'].fillna(0).astype(int)
surv_lmc = merged_lmc.dropna(subset=['time'])
surv_lmc = surv_lmc[surv_lmc['time'] > 0]

print(f"Patients in survival analysis: {len(surv_lmc)}")
print(f"Events (relapses): {surv_lmc['event'].sum()}")

# ── Univariate Cox with rank-based normalisation ──────────────────────
results = []
for lmc in lmc_cols:
    cox_data = surv_lmc[[lmc, 'time', 'event']].dropna().copy()
    n = len(cox_data)
    ranks = rankdata(cox_data[lmc])
    cox_data[f'{lmc}_ranked'] = norm.ppf((ranks - 0.375) / (n + 0.25))

    cph = CoxPHFitter()
    cph.fit(cox_data[[f'{lmc}_ranked', 'time', 'event']],
            duration_col='time', event_col='event')

    summary = cph.summary
    results.append({
        'LMC': lmc,
        'HR': summary.loc[f'{lmc}_ranked', 'exp(coef)'],
        'HR_lower': summary.loc[f'{lmc}_ranked', 'exp(coef) lower 95%'],
        'HR_upper': summary.loc[f'{lmc}_ranked', 'exp(coef) upper 95%'],
        'p_val': summary.loc[f'{lmc}_ranked', 'p']
    })

results_lmc = pd.DataFrame(results)
results_lmc['padj'] = (results_lmc['p_val'] * len(results_lmc)).clip(upper=1.0)
print("\n=== Univariate Cox PH — LMC proportions ===")
print(results_lmc.round(4).to_string(index=False))
results_lmc.to_csv('CoxPH_univariate_LMC.csv', index=False)

In [ ]:
results_lmc_sorted = results_lmc.sort_values('HR', ascending=True)

labels = results_lmc_sorted['LMC'].tolist()
hrs    = results_lmc_sorted['HR'].tolist()
lower  = results_lmc_sorted['HR_lower'].tolist()
upper  = results_lmc_sorted['HR_upper'].tolist()
padj   = results_lmc_sorted['padj'].tolist()

xerr_low  = [hr - lo for hr, lo in zip(hrs, lower)]
xerr_high = [hi - hr for hr, hi in zip(hrs, upper)]

def sig_label(p):
    if p < 0.001:  return '***'
    elif p < 0.01: return '**'
    elif p < 0.05: return '*'
    else:          return 'ns'

sig_labels = [sig_label(p) for p in padj]

colors = []
for hr, p in zip(hrs, padj):
    if p >= 0.05:    colors.append('#999999')
    elif hr > 1:     colors.append('#d62728')
    else:            colors.append('#1f77b4')

fig, ax = plt.subplots(figsize=(9, 7))
y_pos = np.arange(len(labels))

ax.errorbar(
    hrs, y_pos,
    xerr=[xerr_low, xerr_high],
    fmt='o', color='black', ecolor='black',
    elinewidth=1.5, capsize=4, capthick=1.5, markersize=0
)
for i, (hr, y, c) in enumerate(zip(hrs, y_pos, colors)):
    ax.scatter(hr, y, color=c, s=80, zorder=5)

for i, (hr, hi, y, sl) in enumerate(zip(hrs, upper, y_pos, sig_labels)):
    ax.text(hi + 0.02, y, f'  {sl}', va='center', fontsize=10)

ax.axvline(1.0, color='black', linestyle='--', linewidth=1, alpha=0.6)
ax.set_xlim(0.4, 2.1)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels, fontsize=11)
ax.set_xlabel(
    'Hazard Ratio (per 1 SD increase in ranked proportion)\nwith 95% confidence interval',
    fontsize=10
)
ax.set_title(
    'Univariate Cox PH — LMC proportions vs relapse-free survival\n'
    'PPCG cohort (n=670, 158 events) | Bonferroni-adjusted',
    fontsize=11, fontweight='bold'
)
ax.axvspan(0.4, 1.0, alpha=0.04, color='steelblue')
ax.axvspan(1.0, 2.1, alpha=0.04, color='tomato')
ax.text(0.97, 0.55, 'Risk →', transform=ax.transAxes,
        fontsize=9, color='tomato', ha='right', alpha=0.7)
ax.text(0.03, 0.55, '← Protective', transform=ax.transAxes,
        fontsize=9, color='steelblue', ha='left', alpha=0.7)
ax.text(
    0.98, 0.02,
    '*** padj<0.001   ** padj<0.01   * padj<0.05   ns not significant',
    transform=ax.transAxes, fontsize=8, ha='right', va='bottom',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='grey', alpha=0.8)
)

plt.tight_layout()
plt.savefig('ForestPlot_CoxPH_LMC.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved LMC forest plot')

km curves

In [ ]:
lmc_sig = ['LMC2', 'LMC7', 'LMC8', 'LMC13', 'LMC14']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, lmc in enumerate(lmc_sig):
    ax = axes[i]
    median_val = surv_lmc[lmc].median()
    high = surv_lmc[surv_lmc[lmc] >= median_val]
    low  = surv_lmc[surv_lmc[lmc] <  median_val]

    result = logrank_test(
        high['time'], low['time'],
        event_observed_A=high['event'],
        event_observed_B=low['event']
    )
    padj = min(result.p_value * len(lmc_sig), 1.0)

    kmf_high = KaplanMeierFitter()
    kmf_low  = KaplanMeierFitter()

    kmf_high.fit(high['time'], high['event'], label=f'High (n={len(high)})')
    kmf_low.fit(low['time'],   low['event'],  label=f'Low (n={len(low)})')

    kmf_high.plot_survival_function(ax=ax, ci_show=True, color='steelblue')
    kmf_low.plot_survival_function(ax=ax, ci_show=True, color='tomato')

    ax.set_title(f'{lmc}\npadj={padj:.4f}', fontsize=11)
    ax.set_xlabel('Time (days)')
    ax.set_ylabel('Relapse-free survival')
    ax.legend(fontsize=9)

axes[-1].set_visible(False)

plt.suptitle(
    'Relapse-free survival by LMC proportion\n(median split, PPCG cohort | Bonferroni-adjusted)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('KM_curves_LMC.png', dpi=150, bbox_inches='tight')
plt.close()
print('Saved LMC KM curves')